# EtherCAT — one frame through every terminal

A master (`EthercatMaster`) scans a simulated I/O rack on a `VirtualEthercatSegment`: a coupler, 8 digital inputs, 8 digital outputs and 4 analog inputs on a conveyor. For a real line use `UseInterface("eth1")` on a dedicated port (Linux: CAP_NET_RAW; Windows: Npcap). The master stays read-only (SAFE-OP) until `AllowWrites()`.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.28.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.28.0-preview.1"

## Frames and datagrams
An Ethernet frame with EtherType 0x88A4 carries datagrams; each ends with a working counter the slaves increment.

In [ ]:
using System.Buffers.Binary;
using IoTCom.Net;
using IoTCom.Net.Protocols.EtherCat;

var frame = EcatFrame.Encode([
    EcatDatagram.Broadcast(EcatCommand.Brd, EcatRegister.Type, new byte[1]),
    EcatDatagram.Configured(EcatCommand.Fprd, 0x1002, EcatRegister.AlStatus, new byte[2]),
]);
Console.WriteLine(Convert.ToHexString(frame));
foreach (var f in EcatFrame.DescribeFields(frame)) Console.WriteLine($"  {f.Name,-12} {f.Value}");

## Scan the line
A broadcast read counts the slaves, positional writes give them station addresses, and the SII EEPROM tells who they are.

In [ ]:
var rack = new EthercatRackSimulator();
var master = EthercatMaster.Create(o => o.UseLink(rack.Segment));
foreach (var slave in await master.ScanAsync()) Console.WriteLine(slave);

## SAFE-OP: inputs live, outputs safe
The master maps every terminal into one process image (FMMUs) and reads it with one logical datagram per cycle.

In [ ]:
await master.RequestStateAsync(EcatState.SafeOp);
for (var i = 0; i < 20; i++) { rack.Step(0.05); await master.ExchangeAsync(); }
Console.WriteLine($"DI8 {Convert.ToString(master.InputsOf(master.Slaves[1])[0], 2).PadLeft(8, '0')}  (bit 2 e-stop OK, bit 3 guard closed)");
var ai = master.InputsOf(master.Slaves[3]);
Console.WriteLine($"air {BinaryPrimitives.ReadInt16LittleEndian(ai[4..]) / 100.0:0.00} bar, gearbox {BinaryPrimitives.ReadInt16LittleEndian(ai[2..]) / 100.0:0.0} °C");
Console.WriteLine($"CoE 0x1008: {System.Text.Encoding.ASCII.GetString(await master.SdoReadAsync(master.Slaves[3], 0x1008, 0))}");
Console.WriteLine(master.Statistics);
try { await master.RequestStateAsync(EcatState.Op); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }

## OP, a cut cable
A second master with AllowWrites() runs the conveyor; then the cable is cut after the input terminal and the working counter falls short.

In [ ]:
await master.DisposeAsync();
var engineer = EthercatMaster.Create(o => o.UseLink(rack.Segment).AllowWrites());
await engineer.ScanAsync();
await engineer.RequestStateAsync(EcatState.Op);
engineer.SetOutputBit(engineer.Slaves[2], 0, true);   // conveyor motor
for (var i = 0; i < 300; i++) { rack.Step(0.01); await engineer.ExchangeAsync(); }
var analog = engineer.InputsOf(engineer.Slaves[3]);
Console.WriteLine($"belt {BinaryPrimitives.ReadInt16LittleEndian(analog[6..]) / 100.0:0.0} m/min, wkc {engineer.Statistics.LastWorkingCounter}/{engineer.Statistics.ExpectedWorkingCounter}");
rack.Segment.BreakAfter = 2;
await engineer.ExchangeAsync();
Console.WriteLine($"cable cut: wkc {engineer.Statistics.LastWorkingCounter}/{engineer.Statistics.ExpectedWorkingCounter}");
rack.Segment.BreakAfter = null;
await engineer.DisposeAsync();
Console.WriteLine($"after closing the master: DO8 is in {EcatAlStatus.Name(rack.DigitalOut.State)}");

## Going further
`iotcom ethercat interfaces`, `iotcom ethercat scan --if <port>`, `iotcom ethercat watch --sim`, `iotcom ethercat sdo 3 1008:00 --sim`, `iotcom ethercat demo`, the Gallery's *I/O rack over EtherCAT* and the EthercatRack sample. See `docs/en/protocols/ethercat.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*